In [1]:
# Compatibility helper for notebook-side evaluation patching.
# Some Kaggle eval-patch snippets reference `import_line` before it is assigned.
import_line = None


# PROB IP102 Continual Learning Evaluation & Training Pipeline

Notebook này cung cấp quy trình (pipeline) hoàn chỉnh để chạy đánh giá và huấn luyện mô hình **PROB** (Probabilistic Open-World Object Detection) với bộ dữ liệu sâu bệnh **IP102** trên **Kaggle** sử dụng **2 GPU T4 (DDP)**.

### Hướng dẫn chuẩn bị trước khi chạy:
1. Đảm bảo Notebook này được cấu hình chạy trên **Accelerator: GPU T4 x2** (ở panel Settings phía bên phải).
2. Đảm bảo **Internet** được bật (Internet on).
3. Thêm các Dataset sau vào Notebook:
   - Tập dữ liệu ảnh IP102 (đường dẫn: `/kaggle/input/datasets/rtlmhjbn/ip02-dataset`).
   - Tập nhãn COCO của IP102 (đường dẫn: `/kaggle/input/datasets/eljazouly/ip102-coco-annotations`).
   - Thư mục chứa các checkpoint pretrain tác giả hoặc của bạn (ví dụ: `prob-checkpoints` chứa `t1.pth`, `t2.pth`, `t3.pth`, `t4.pth`).

## Bước 1: Clone Repository và Biên dịch thư viện CUDA

In [2]:
# Clone repo chứa mã nguồn tùy chỉnh đã được cấu hình lớp IP102
# Chạy trên Kaggle: clone vào /kaggle/working để tránh các path local-only.
!bash -lc 'if [ ! -d /kaggle/working/Prob-ann-custom ]; then git clone https://github.com/nta2112/Prob-ann-custom.git /kaggle/working/Prob-ann-custom; else cd /kaggle/working/Prob-ann-custom && git pull; fi'
%cd /kaggle/working/Prob-ann-custom


Cloning into '/kaggle/working/Prob-ann-custom'...
remote: Enumerating objects: 244, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 244 (delta 29), reused 33 (delta 17), pack-reused 193 (from 1)
Receiving objects: 100% (244/244), 61.35 MiB | 47.63 MiB/s, done.
Resolving deltas: 100% (98/98), done.
/kaggle/working/Prob-ann-custom


In [3]:
# Cài đặt các thư viện yêu cầu theo môi trường Kaggle
%pip install -q -r requirements.txt
%pip install -q pycocotools einops

# Biên dịch các toán tử CUDA MultiScaleDeformableAttention
%cd /kaggle/working/Prob-ann-custom/models/ops
!python setup.py build install
%cd /kaggle/working/Prob-ann-custom


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 66.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
/kaggle/working/Prob-ann-custom/models/ops
running build
running build_py
creating build/lib.

## Bước 2: Thiết lập dữ liệu IP102 (Chuyển đổi sang định dạng VOC XML)

Mã nguồn PROB sử dụng dataloader định dạng VOC XML. Lệnh dưới đây chạy script `setup_ip102.py` để quét ảnh, tự động vá file cấu hình lớp `open_world.py` và chuyển nhãn từ COCO JSON sang VOC XML.

In [4]:
# Cấu hình đường dẫn dữ liệu chuẩn của IP102 trên Kaggle
JSON_DIR = "/kaggle/input/datasets/eljazouly/ip102-coco-annotations/coco_annotations"
IMAGE_DIR = "/kaggle/input/datasets/rtlmhjbn/ip02-dataset"
OUTPUT_DIR = "/kaggle/working/datasets/IP102"

!python tools/setup_ip102.py \
    --json_dir {JSON_DIR} \
    --image_dir {IMAGE_DIR} \
    --output_dir {OUTPUT_DIR}

Total IP102 Classes: 102
Scanning for images in /kaggle/input/datasets/rtlmhjbn/ip02-dataset...
Found 75222 image files in the dataset path.
Patched open_world.py successfully with IP102 class lists!
Converting Train JSON...
Converted 45095 train images.
Converting Val JSON...
Converted 7508 val images.
Converting Test JSON...
Converted 22619 test images.
Splitting train images into tasks...
Generated ImageSets splits under /kaggle/working/datasets/IP102/ImageSets/IP102:
  owod_t1_train.txt: 13839 images
  owod_t2_train.txt: 9510 images
  owod_t3_train.txt: 12837 images
  owod_t4_train.txt: 8909 images
  owod_all_task_test.txt: 22619 images
Dataset setup for IP102 completed successfully!


## Bước 3: Chạy huấn luyện thử nghiệm 1 Epoch & Đánh giá xen kẽ cho từng Task (Task 1 -> Task 4)

Lệnh dưới đây chạy kịch bản `run_train_eval_kaggle.sh` để thực hiện **huấn luyện tinh chỉnh 1 Epoch rồi chạy đánh giá (test) ngay lập tức** cho từng task liên tiếp:
- Tham số thứ nhất: Đường dẫn thư mục chứa checkpoint ban đầu (`/kaggle/input/prob-checkpoints`)
- Tham số thứ hai: Batch size huấn luyện trên mỗi GPU (khuyến nghị: `2` để tránh OOM)
- Tham số thứ ba: Batch size đánh giá trên mỗi GPU (khuyến nghị: `4` để chạy ổn định hơn)
- Tham số thứ tư: Số process phân tán trên mỗi node (khuyến nghị: `1` để tránh DDP bị kill trên Kaggle)

**Quy trình thực thi:**
1. **Task 1:** Huấn luyện 1 Epoch (từ checkpoint pretrain `t1.pth`) -> Đánh giá kiểm thử ngay lập tức trên lớp 1-27.
2. **Task 2:** Huấn luyện 1 Epoch (từ checkpoint của Task 1) -> Tinh chỉnh mẫu 1 Epoch -> Đánh giá kiểm thử ngay lập tức trên lớp 1-52.
3. **Task 3:** Huấn luyện 1 Epoch (từ checkpoint của Task 2) -> Tinh chỉnh mẫu 1 Epoch -> Đánh giá kiểm thử ngay lập tức trên lớp 1-77.
4. **Task 4:** Huấn luyện 1 Epoch (từ checkpoint của Task 3) -> Tinh chỉnh mẫu 1 Epoch -> Đánh giá kiểm thử ngay lập tức trên lớp 1-102.

In [ ]:
# Chạy quy trình xen kẽ Huấn luyện 1 Epoch -> Đánh giá kiểm thử cho từng Task
# Dùng batch an toàn hơn trên Kaggle T4 x2 và tắt DDP đa process để tránh SIGTERM.
!bash tools/run_train_eval_kaggle.sh /kaggle/input/prob-checkpoints 2 4 1


Searching for t1.pth in /kaggle/input...
Found checkpoint directory: /kaggle/input/models/chienkhu/pretrain-prob/pytorch/default/1/MOWODB
BẮT ĐẦU QUY TRÌNH HUẤN LUYỆN 1 EPOCH & ĐÁNH GIÁ TỪNG TASK (TASK 1 -> TASK 4)
Checkpoints Directory: /kaggle/input/models/chienkhu/pretrain-prob/pytorch/default/1/MOWODB
Train Batch Size per GPU: 2
Eval Batch Size per GPU: 4
Data Root: /kaggle/working/datasets/IP102
W&B Mode: disabled
----------------------------------------------------
>>> [TASK 1] Huấn luyện 1 Epoch (Epoch 0)

*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
{'OWDETR': ('aeroplane', 'bicycle', 'bird', 'boat', 'bus', 'car', 'cat', 'cow', 'dog', 'horse', 'motorbike', 'sheep', 'train', 'elephant', 'bear', 'zebra', 'giraffe', 'truck', 'person', '

## Bước 4: Chạy Đánh giá trực tiếp (Nếu checkpoint đã tương thích sẵn)

Chỉ sử dụng lệnh này nếu Sans đã có sẵn các checkpoint đã được huấn luyện sẵn trên IP102 (đầy đủ 103 lớp đầu ra). Lệnh này sẽ bỏ qua huấn luyện và chạy đánh giá test trực tiếp trên cả 4 task bằng 2 GPU.

In [6]:
# Chạy đánh giá trực tiếp với batch size phù hợp Kaggle
# !bash tools/run_eval_kaggle.sh /kaggle/input/prob-checkpoints 8
